In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import joblib

sys.path.append(os.path.abspath('..'))
from src.transformadores import LimpiadorFormatosKaggle, CalculadorDistancia
from src.pipeline import construir_pipeline_preprocesamiento

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# 1. Cargar datos
df_train = pd.read_csv('../data/processed/train_processed.csv')
df_test = pd.read_csv('../data/processed/test_processed.csv')

# 2. Aplicar Transformadores Base
limpiador = LimpiadorFormatosKaggle()
calculador = CalculadorDistancia()

df_train_final = calculador.transform(limpiador.transform(df_train))
df_test_final = calculador.transform(limpiador.transform(df_test))

# 3. Separar X e Y (restaurando la limpieza Regex en la etiqueta objetivo)
X_train = df_train_final.drop(columns=['Time_taken(min)'])
y_train = df_train_final['Time_taken(min)'].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)

X_test = df_test_final.drop(columns=['Time_taken(min)'])
y_test = df_test_final['Time_taken(min)'].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)

In [ ]:
# 4. Crear el pipeline integrado con el Árbol de Decisión
pipeline_arbol = Pipeline(steps=[
    ('preprocesador', construir_pipeline_preprocesamiento()),
    ('modelo_arbol', DecisionTreeRegressor(max_depth=10, min_samples_split=5, random_state=42))
])

print("Entrenando Árbol de Decisión...")
pipeline_arbol.fit(X_train, y_train)

# 5. Evaluación en Test Set
y_pred_test_arbol = pipeline_arbol.predict(X_test)
mae_arbol = mean_absolute_error(y_test, y_pred_test_arbol)
r2_arbol = r2_score(y_test, y_pred_test_arbol)

print(f"MAE (Test): {mae_arbol:.2f} minutos")
print(f"R2 (Test): {r2_arbol:.4f}")

In [ ]:
print("\nEjecutando K-Fold Cross Validation (K=5)...")
scores_mae_arbol = -cross_val_score(pipeline_arbol, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')

print(f"MAE Promedio K-Fold: {scores_mae_arbol.mean():.2f} minutos (+/- {scores_mae_arbol.std():.2f} min)")

In [ ]:
# Guardar el modelo basado en árboles
ruta_modelo_arbol = os.path.join('..', 'models', 'modelo_arbol_decision.joblib')
joblib.dump(pipeline_arbol, ruta_modelo_arbol)
print(f"Nuevo modelo serializado en: {ruta_modelo_arbol}")